# Lesson 2 — vectors
A vector is a list of floats, same length every row. Search returns rows plus a `_distance` column.

In [1]:
%pip install -q lancedb pandas

/opt/Code/github.com/Soul-Brews-Studio/lancedb-oracle/lessons/01-basics/.venv/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import lancedb

db = lancedb.connect("./data/lesson2")

In [3]:
rows = [
    {"id": "a", "vector": [1.0, 0.0, 0.0]},
    {"id": "b", "vector": [0.0, 1.0, 0.0]},
    {"id": "c", "vector": [0.0, 0.0, 1.0]},
    {"id": "d", "vector": [0.9, 0.1, 0.0]},  # close to a
    {"id": "e", "vector": [0.5, 0.5, 0.0]},  # between a and b
]
tbl = db.create_table("points", data=rows, mode="overwrite")
tbl.schema  # vector: fixed_size_list<float>[3]

id: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float

In [4]:
tbl.to_pandas()

,id,vector
0,a,"[1.0, 0.0, 0.0]"
1,b,"[0.0, 1.0, 0.0]"
2,c,"[0.0, 0.0, 1.0]"
3,d,"[0.9, 0.1, 0.0]"
4,e,"[0.5, 0.5, 0.0]"


In [5]:
# Which row is nearest to [1, 0, 0]?
q = [1.0, 0.0, 0.0]
tbl.search(q).limit(3).to_pandas()
# _distance = squared L2 by default. a=0, d=0.02, e=0.5

,id,vector,_distance
0,a,"[1.0, 0.0, 0.0]",0.00
1,d,"[0.9, 0.1, 0.0]",0.02
2,e,"[0.5, 0.5, 0.0]",0.50


In [6]:
# Same query, cosine metric. Direction only, length ignored.
tbl.search(q).metric("cosine").limit(3).to_pandas()

,id,vector,_distance
0,a,"[1.0, 0.0, 0.0]",0.000000
1,d,"[0.9, 0.1, 0.0]",0.006116
2,e,"[0.5, 0.5, 0.0]",0.292893
